# Digikala Products: Vectorization and Search

This notebook produces `products.npz` with 1,283,496 source row indices and 768-dimensional Yalda vectors. To search the released dataset, run the steps in section 2.

- `use` is the default mode for searching the release and retrieving metadata from the original CSV.
- `tokenize` builds the token cache from the original CSV and saves it to Google Drive.
- `build` reads the cache and uses a Colab GPU to build or resume the vector file.

[Download the vectorized products dataset](https://drive.google.com/file/d/1H9YIchZXzkdQXKIQCXMvryQ3TNPrshV2/view?usp=sharing)

## 1. Install dependencies and set the run mode

To run the search example, keep `MODE = "use"`, set `RUN_SEARCH_EXAMPLE = True`, and run the cells in order. Colab can download the NPZ into its cloud runtime. If you have a copy in Google Drive, set `VECTOR_FILE` to its path and enable `MOUNT_DRIVE_FOR_USAGE`. Use the mounted path if you hit Drive's download quota.

In [ ]:
%pip -q install "transformers==4.57.1" "sentence-transformers==5.1.2" "huggingface-hub>=0.34,<1" "gdown==6.1.1" psutil

In [ ]:
MODE = 'use'  # use, tokenize, or build
RUN_SEARCH_EXAMPLE = False
VECTOR_FILE = ''  # blank: use the published Google Drive file
MOUNT_DRIVE_FOR_USAGE = False
SOURCE_CSV = ''  # blank: use the pinned original CSV from Hugging Face

DATASET_LINK = 'https://drive.google.com/file/d/1H9YIchZXzkdQXKIQCXMvryQ3TNPrshV2/view?usp=sharing'
QUERIES = ['گوشی موبایل سامسونگ', 'کیف مدرسه', 'کفش ورزشی برای پیاده\u200cروی', 'تابه نچسب', 'شارژر همراه برای سفر']
DISPLAY_COLUMNS = ['id', 'title_fa', 'Brand', 'Category1', 'Category2', 'sub_category', 'Price', 'Rate', 'Rate_cnt']

KIND = 'products'
TOKEN_SHARE_LINK = ''  # optional published token ZIP link
LOCAL_TOKEN_ZIP = ''  # optional: mounted Drive ZIP path, instead of public link
OUTPUT_ROOT = '/content/drive/MyDrive/digikala_yalda_final_v1'
CACHE_TOKENS_IN_RAM = True
GLOBAL_DEDUP = True
RAM_RESERVE_GIB = 2.5
KEEP_CHECKPOINTS = False  # delete only after final NPZ is verified on Drive

if MODE not in ('use', 'tokenize', 'build'):
    raise ValueError("MODE must be 'use', 'tokenize', or 'build'.")

### Shared settings and text normalization

The notebook pins the model and source CSV revisions. Documents and queries use the same rules for letters, digits, spaces, zero-width non-joiners and blank values. To redisplay or reconstruct document text, the notebook reads the original CSV columns.

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'true'
os.environ['RAYON_NUM_THREADS'] = str(min(8, len(os.sched_getaffinity(0))))
import json, time, shutil, hashlib, tempfile, ast, re, unicodedata, io, zipfile, subprocess, psutil, math
from pathlib import Path
from functools import lru_cache
from importlib.metadata import version
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
ROOT = Path(OUTPUT_ROOT)
LOCAL = Path('/content/digikala_yalda_split_' + KIND)
MODEL_ID = 'Yaldat/Yalda-Embedding'
MODEL_REVISION = '998d1710853957d4108eff6ebebdc9474d986552'
DATASET_ID = 'RadeAI/Digikala_comments_products'
REVISION = '89c3133b169c8d3793db8834f56f32fee33d9db0'
SPECS = {
    'products': {'file':'digikala-products.csv', 'fields':['title_fa','Brand','Category1','Category2','sub_category'], 'labels':['عنوان','برند','دسته','زیردسته','گروه'], 'rows':1283496},
    'comments': {'file':'digikala-comments.csv', 'fields':['title','body','advantages','disadvantages'], 'labels':['عنوان','نظر','نقاط قوت','نقاط ضعف'], 'rows':6156289},
}
MAX_TOKENS = {'products':128, 'comments':512}
REMOVE_PRODUCT_LABELS = True
SHARD_ROWS = 8192
TOKEN_BATCH = 4096
DIM = 768
RUN_DATASETS = (KIND,)
CSV_OPTIONS = dict(dtype="str", keep_default_na=False, encoding='utf-8-sig', skip_blank_lines=True, on_bad_lines='error')
DOCUMENT_PROMPT = ''
QUERY_PROMPT = 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:'
TOKEN_CONFIG = dict(format_version=1, model=MODEL_ID, model_revision=MODEL_REVISION, dataset=DATASET_ID,
    dataset_revision=REVISION, specs=SPECS, max_tokens=MAX_TOKENS, remove_product_labels=REMOVE_PRODUCT_LABELS,
    stopwords=False, normalization='pilot_persian_v1', document_prompt=DOCUMENT_PROMPT, query_prompt=QUERY_PROMPT,
    lower_case=False, add_special_tokens=True, padding_side='left', truncation_side='right', shard_rows=SHARD_ROWS,
    index='zero-based parsed CSV data row', csv_options=CSV_OPTIONS,
    tokenizer_versions={p:version(p) for p in ['transformers','tokenizers']})
TOKEN_ROOT = ROOT/'tokens'
VECTOR_ROOT = ROOT/'vectors'

def digest(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False).encode()).hexdigest()

def write_json(path,obj):
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8')
    tmp.replace(path)

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def ensure_manifest(folder, config, create=True):
    folder.mkdir(parents=True,exist_ok=True)
    path=folder/'manifest.json'
    expected={'config':config,'signature':digest(config)}
    if path.exists():
        if read_json(path)!=expected:
            raise RuntimeError(f'Incompatible manifest: {path}. Use a new ROOT for changed settings.')
    elif not create or any(folder.glob('*/part_*.npz')):
        raise RuntimeError(f'Missing manifest: {path}')
    else:
        write_json(path,expected)
    return expected['signature']

def intervals(kind):
    total=SPECS[kind]['rows']
    return [(start,min(SHARD_ROWS,total-start)) for start in range(0,total,SHARD_ROWS)]

def part_path(folder,start):
    return folder/f'part_{start:09d}.npz'

def check_parts(folder,kind):
    expected={part_path(folder,s).name for s,n in intervals(kind)}
    actual={p.name for p in folder.glob('part_*.npz')}
    if expected!=actual:
        raise RuntimeError(f'{kind}: missing/extra parts: {list(expected^actual)[:10]}')

def publish(path,arrays,validator,start,size):
    # Write and check locally, then publish only a complete file on Drive.
    with tempfile.TemporaryDirectory(dir=LOCAL) as tmpdir:
        local=Path(tmpdir)/'part.npz'
        np.savez(local,**arrays)
        validator(local,start,size)
        tmp=path.with_suffix('.npz.tmp')
        shutil.copyfile(local,tmp)
        if tmp.stat().st_size!=local.stat().st_size:
            raise IOError('Incomplete Drive copy')
        tmp.replace(path)

def get_source(kind):
    return hf_hub_download(repo_id=DATASET_ID,repo_type='dataset',filename=SPECS[kind]['file'],revision=REVISION)

def csv_header(path):
    raw=pd.read_csv(path,nrows=0,**CSV_OPTIONS).columns.tolist()
    clean=[s.strip() for s in raw]
    if len(set(clean))!=len(clean): raise ValueError('Duplicate stripped headers')
    return dict(zip(clean,raw))

def token_arrays(path,start,size,limit):
    with np.load(path,allow_pickle=False) as d:
        if set(d.files)!={'source_row','input_ids','offsets','valid'}: raise ValueError('Invalid token schema')
        rows,ids,offsets,valid=(d[k] for k in ('source_row','input_ids','offsets','valid'))
    if rows.dtype!=np.int64 or not np.array_equal(rows,np.arange(start,start+size,dtype=np.int64)): raise ValueError('Invalid source rows')
    if ids.ndim!=1 or ids.dtype!=np.uint32 or offsets.dtype!=np.int32 or offsets.shape!=(size+1,): raise ValueError('Invalid ragged arrays')
    lengths=np.diff(offsets.astype(np.int64))
    if offsets[0]!=0 or offsets[-1]!=len(ids) or np.any(lengths<0) or np.any(lengths>limit): raise ValueError('Invalid offsets or lengths')
    if valid.dtype!=np.bool_ or not np.array_equal(valid,lengths>0): raise ValueError('Invalid validity mask')
    return ids,offsets,valid

def available_ram():
    # psutil plus cgroup limit if the runtime exposes it.
    available = int(psutil.virtual_memory().available)
    for limit_name, usage_name in [('/sys/fs/cgroup/memory.max','/sys/fs/cgroup/memory.current'),
                                  ('/sys/fs/cgroup/memory/memory.limit_in_bytes','/sys/fs/cgroup/memory/memory.usage_in_bytes')]:
        try:
            limit = Path(limit_name).read_text().strip()
            if limit != 'max' and int(limit) < 2**60:
                used=int(Path(usage_name).read_text().strip())
                # Inactive file cache can be reclaimed (especially our local mmap).
                try:
                    stats=dict(line.split() for line in Path(usage_name).with_name('memory.stat').read_text().splitlines())
                    used=max(0,used-int(stats.get('total_inactive_file',stats.get('inactive_file',0))))
                except (OSError,ValueError):
                    pass
                available = min(available, max(0,int(limit)-used))
        except (OSError, ValueError):
            pass
    return available

RAM_RESERVE = int(RAM_RESERVE_GIB * 2**30)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):
            h.update(block)
    return h.hexdigest()

In [ ]:
translation = str.maketrans("يكى٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "یکی01234567890123456789")
empty_markers = {"", "nan", "none", "null", "<na>", "[]"}

def normalize(value):
    if value is None or pd.isna(value):
        return ""
    text = unicodedata.normalize("NFC", str(value)).translate(translation)
    text = re.sub(r"[\u200b\u200d\u200e\u200f\u202a-\u202e\u2060\u2066-\u2069\ufeff]", "", text)
    text = re.sub(r"[\u0000-\u001f\u007f]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r" *\u200c+ *", "\u200c", text).strip("\u200c ")
    return "" if text.casefold() in empty_markers else text

def normalize_list(value):
    text = normalize(value)
    if text.startswith(("[", "(")):
        try:
            items = ast.literal_eval(text)
            if isinstance(items, (list, tuple)):
                return "؛ ".join(part for item in items if (part := normalize(item)))
        except (ValueError, SyntaxError):
            pass
    return text

assert normalize("  كيفيت\u200f  كالا  خوب نيست ") == "کیفیت کالا خوب نیست"
assert normalize_list("['خوب', 'ارزان']") == "خوب؛ ارزان"



@lru_cache(maxsize=32768)
def normalize_category(value):
    return normalize(value)


def chunk_texts(frame, kind):
    spec = SPECS[kind]
    fields, labels = spec['fields'], spec['labels']
    texts = []
    if kind == 'products':
        for values in frame[fields].itertuples(index=False, name=None):
            parts = []
            for i, (value, label) in enumerate(zip(values, labels)):
                value = normalize(value) if i == 0 else normalize_category(value)
                if value:
                    parts.append(value if REMOVE_PRODUCT_LABELS else f"{label}: {value}")
            texts.append(" | ".join(parts))
    else:
        for values in frame[fields].itertuples(index=False, name=None):
            parts = []
            for i, (value, label) in enumerate(zip(values, labels)):
                value = normalize_list(value) if i >= 2 else normalize(value)
                if value:
                    parts.append(f"{label}: {value}")
            texts.append(" | ".join(parts))
    return texts

## 2. Search the released dataset

`get_vector_file()` opens the released NPZ or the path you provide. `vector_blocks()` reads the file block by block and yields valid rows. `search()` embeds each query with Yalda's official `query` prompt and a 128-token limit, reconstructs document vectors using their row scales and ranks results by cosine similarity. Ties are ordered by ascending source row.

`lookup_metadata()` reads the original CSV columns in result order. `source_row` is zero-based and follows the parsed CSV data rows after the header and skipped blank lines. If you set `SOURCE_CSV`, use the same CSV revision, row order and format. `embedding_text` displays the prepared document text before applying the token limit.

In [ ]:
from contextlib import ExitStack

def get_vector_file():
    if MOUNT_DRIVE_FOR_USAGE:
        from google.colab import drive
        drive.mount('/content/drive')

    if VECTOR_FILE:
        path = Path(VECTOR_FILE).expanduser()
        if not path.is_file():
            raise FileNotFoundError(path)
        return path

    cache = Path('/content') if Path('/content').is_dir() else Path.cwd()
    path = cache / f'{KIND}.npz'
    if not path.is_file():
        partial = path.with_suffix('.npz.download')
        subprocess.run(
            ['gdown', '--continue', '--retries', '3', DATASET_LINK, '-O', str(partial)],
            check=True,
        )
        if not zipfile.is_zipfile(partial):
            raise ValueError('Expected a released NPZ file.')
        partial.replace(path)
    return path


def vector_blocks(path, block_rows=8192):
    if block_rows < 1:
        raise ValueError('block_rows must be positive.')
    total = SPECS[KIND]['rows']
    schema = {
        'source_row': ((total,), np.dtype('int64')),
        'vectors': ((total, DIM), np.dtype('int8')),
        'scales': ((total,), np.dtype('float32')),
        'valid': ((total,), np.dtype('bool')),
    }
    readers = {
        (1, 0): np.lib.format.read_array_header_1_0,
        (2, 0): np.lib.format.read_array_header_2_0,
    }
    with zipfile.ZipFile(path) as archive, ExitStack() as stack:
        if sorted(archive.namelist()) != sorted(f'{key}.npy' for key in schema):
            raise ValueError('Unexpected NPZ schema. Use the released dataset for this notebook.')

        streams = {}
        for key, (expected_shape, expected_dtype) in schema.items():
            stream = stack.enter_context(archive.open(f'{key}.npy'))
            version = np.lib.format.read_magic(stream)
            if version not in readers:
                raise ValueError(f'Unsupported NPY version for {key}: {version}')
            shape, fortran_order, dtype = readers[version](stream)
            if shape != expected_shape or dtype != expected_dtype or fortran_order:
                raise ValueError(f'Unexpected shape, dtype or layout for {key}.')
            streams[key] = stream

        for start in range(0, total, block_rows):
            size = min(block_rows, total - start)
            block = {}
            for key, (shape, dtype) in schema.items():
                tail = shape[1:]
                nbytes = size * math.prod(tail) * dtype.itemsize
                raw = streams[key].read(nbytes)
                if len(raw) != nbytes:
                    raise EOFError(f'Incomplete NPZ member: {key}')
                block[key] = np.frombuffer(raw, dtype=dtype).reshape((size, *tail))

            rows = block['source_row']
            if not np.array_equal(rows, np.arange(start, start + size, dtype=np.int64)):
                raise ValueError('Unexpected source_row order.')
            valid = block['valid']
            yield rows[valid], block['vectors'][valid], block['scales'][valid]

        for key, stream in streams.items():
            if stream.read(1):
                raise ValueError(f'Trailing bytes in NPZ member: {key}')


def lookup_metadata(source_rows, columns=None):
    wanted = [int(row) for row in source_rows]
    if not wanted:
        return pd.DataFrame()
    if min(wanted) < 0 or max(wanted) >= SPECS[KIND]['rows']:
        raise IndexError('source_row is outside the original CSV.')

    source = Path(SOURCE_CSV) if SOURCE_CSV else get_source(KIND)
    usecols = None
    if columns is not None:
        headers = csv_header(source)
        usecols = [headers[column] for column in columns]

    targets = np.asarray(sorted(set(wanted)), dtype=np.int64)
    next_target = start = 0
    parts = []
    with pd.read_csv(source, chunksize=SHARD_ROWS, usecols=usecols, **CSV_OPTIONS) as reader:
        for frame in reader:
            end = start + len(frame)
            stop = targets.searchsorted(end)
            if stop > next_target:
                hits = targets[next_target:stop]
                part = frame.iloc[hits - start].copy()
                part.columns = part.columns.str.strip()
                part.insert(0, 'source_row', hits)
                parts.append(part)
                next_target = stop
            if next_target == len(targets):
                break
            start = end

    if next_target != len(targets):
        raise IndexError('Some source rows were not found in the original CSV.')
    return pd.concat(parts).set_index('source_row').loc[wanted].reset_index()


def normalize_vectors(vectors):
    vectors = np.asarray(vectors, dtype=np.float32)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    if not np.isfinite(norms).all() or np.any(norms <= 1e-12):
        raise ValueError('Expected finite, nonzero embeddings.')
    return vectors / norms


def load_query_model():
    import torch
    from sentence_transformers import SentenceTransformer

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    precision = torch.float16 if device == 'cuda' else torch.float32
    model = SentenceTransformer(
        MODEL_ID, revision=MODEL_REVISION, device=device,
        model_kwargs={'torch_dtype': precision},
    )
    model.eval()
    model.max_seq_length = MAX_TOKENS[KIND]
    model.tokenizer.padding_side = 'left'
    assert model.tokenizer.truncation_side == 'right'
    assert model.prompts['query'] == QUERY_PROMPT
    assert model.get_sentence_embedding_dimension() == 1024
    return model


def search(path, model, queries, top_k=5):
    texts = [normalize(query) for query in queries]
    if top_k < 1 or not texts or not all(texts):
        raise ValueError('Provide nonempty queries and a positive top_k.')

    query_vectors = model.encode(
        texts, prompt_name='query', batch_size=16, convert_to_numpy=True,
        normalize_embeddings=False, show_progress_bar=False,
    )
    query_vectors = normalize_vectors(query_vectors[:, :DIM])
    best = [[] for query in texts]

    for rows, codes, scales in vector_blocks(path):
        if not len(rows):
            continue
        vectors = normalize_vectors(codes.astype(np.float32) * scales[:, None])
        scores = vectors @ query_vectors.T
        for i, previous in enumerate(best):
            order = np.lexsort((rows, -scores[:, i]))[:top_k]
            hits = [(float(scores[j, i]), int(rows[j])) for j in order]
            best[i] = sorted(previous + hits, key=lambda hit: (-hit[0], hit[1]))[:top_k]
    return best


def show_results(queries, results):
    from IPython.display import display

    rows = sorted({row for group in results for _, row in group})
    if not rows:
        print('No valid results.')
        return
    metadata = lookup_metadata(rows).set_index('source_row')
    metadata['embedding_text'] = chunk_texts(metadata, KIND)

    for query, group in zip(queries, results):
        table = metadata.loc[[row for _, row in group]].reset_index()
        table.insert(1, 'cosine_similarity', [score for score, _ in group])
        first = ['source_row', 'cosine_similarity', 'embedding_text']
        first += [c for c in DISPLAY_COLUMNS if c in table.columns]
        table = table[first + [c for c in table if c not in first]]
        print(query)
        display(table)

In [ ]:
if MODE == 'use' and RUN_SEARCH_EXAMPLE:
    vector_file = get_vector_file()
    query_model = load_query_model()
    results = search(vector_file, query_model, QUERIES, top_k=5)
    show_results(QUERIES, results)

### Colab and the language-model API

Yalda embeds queries and searches the data in the runtime without an API key. Requests to the project's downstream language-model API must originate from an Iranian IP address. If Colab cannot reach the gateway, make the API request from the team's environment with an Iranian IP address.

## 3. Reproduce the vector file in Colab

Run `MODE = "tokenize"` in a CPU runtime to build the cache from the pinned CSV. It writes `products_tokens.zip` to `OUTPUT_ROOT`. If a compatible cache already exists, provide its mounted path in `LOCAL_TOKEN_ZIP` or its ZIP link in `TOKEN_SHARE_LINK`.

Switch to a GPU runtime and run `MODE = "build"` with the same `OUTPUT_ROOT`. Use one writer per output directory and allow space for the intermediate checkpoints and the final file.

In [ ]:
if MODE in ('tokenize', 'build'):
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT.mkdir(parents=True, exist_ok=True)
    LOCAL.mkdir(parents=True, exist_ok=True)

### Build and save the token cache

In [ ]:
if MODE == 'tokenize':
    from transformers import AutoTokenizer
    TOKEN_SIGNATURE=ensure_manifest(TOKEN_ROOT,TOKEN_CONFIG)
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION,use_fast=True)
    assert tokenizer.is_fast
    assert tokenizer.truncation_side=='right', tokenizer.truncation_side
    tokenizer.padding_side='left'
    assert len(tokenizer)<2**32
    print('CPU threads:',os.environ['RAYON_NUM_THREADS'])
    print('Drive free GiB (mount estimate):',round(shutil.disk_usage(ROOT).free/2**30,2))
    print('Token output:',TOKEN_ROOT)

    # All records kept, even duplicates and blank documents. Only tokenization work is deduplicated.
    def tokenize_shard(texts,limit):
        unique=list(dict.fromkeys(t for t in texts if t))
        encoded={}
        for begin in range(0,len(unique),TOKEN_BATCH):
            batch=unique[begin:begin+TOKEN_BATCH]
            result=tokenizer(batch,add_special_tokens=True,padding=False,truncation=True,max_length=limit,
                             return_attention_mask=False,return_token_type_ids=False,verbose=False)['input_ids']
            encoded.update(zip(batch,result))
        sequences=[encoded[t] if t else [] for t in texts]
        lengths=np.fromiter(map(len,sequences),dtype=np.int32,count=len(texts))
        offsets=np.empty(len(texts)+1,dtype=np.int32); offsets[0]=0
        np.cumsum(lengths,out=offsets[1:])
        ids=np.empty(int(offsets[-1]),dtype=np.uint32)
        for i,seq in enumerate(sequences):
            ids[offsets[i]:offsets[i+1]]=seq
        return ids,offsets,lengths>0,len(unique)

    def process_tokens(kind):
        folder=TOKEN_ROOT/kind; folder.mkdir(exist_ok=True)
        marker=folder/'complete.json'
        marker.unlink(missing_ok=True)
        source=get_source(kind)
        headers=csv_header(source); fields=SPECS[kind]['fields']; limit=MAX_TOKENS[kind]
        if not set(fields).issubset(headers): raise ValueError(f'Missing fields: {set(fields)-headers.keys()}')
        expected=SPECS[kind]['rows']; start=0; count=valid_count=new=reused=0
        prep=tok=save=0.; unique_total=0; t0=time.perf_counter()
        validator=lambda p,s,n:token_arrays(p,s,n,limit)
        with pd.read_csv(source,usecols=[headers[f] for f in fields],chunksize=SHARD_ROWS,**CSV_OPTIONS) as reader, tqdm(total=expected,desc='Tokenize '+kind,unit='rows') as bar:
            for frame in reader:
                size=len(frame); target=part_path(folder,start)
                if start+size>expected or size!=min(SHARD_ROWS,expected-start): raise RuntimeError('Unexpected CSV length')
                cached=None
                if target.exists():
                    try: cached=validator(target,start,size)
                    except Exception as exc: print('Rebuilding',target.name,str(exc))
                if cached is None:
                    t=time.perf_counter(); frame.columns=frame.columns.str.strip(); texts=chunk_texts(frame,kind); prep+=time.perf_counter()-t
                    t=time.perf_counter(); ids,offsets,valid,unique=tokenize_shard(texts,limit); tok+=time.perf_counter()-t
                    t=time.perf_counter(); publish(target,dict(source_row=np.arange(start,start+size,dtype=np.int64),input_ids=ids,offsets=offsets,valid=valid),validator,start,size); save+=time.perf_counter()-t
                    new+=size; unique_total+=unique
                else:
                    ids,offsets,valid=cached; reused+=size
                count+=len(ids); valid_count+=int(valid.sum()); start+=size
                bar.update(size); bar.set_postfix(new=new,resumed=reused,token_rows_s=round(new/max(prep+tok+save,1e-9),1))
        if start!=expected: raise RuntimeError(f'Expected {expected}, got {start}')
        check_parts(folder,kind)
        report=dict(signature=TOKEN_SIGNATURE,kind=kind,total_rows=start,valid_rows=valid_count,total_tokens=count,
                    new_rows=new,resumed_rows=reused,unique_texts_tokenized=unique_total,preprocessing_seconds=prep,
                    tokenization_seconds=tok,storage_seconds=save,wall_seconds=time.perf_counter()-t0,
                    bytes=sum(p.stat().st_size for p in folder.glob('part_*.npz')),complete=True)
        write_json(marker,report); print(json.dumps(report,indent=2))


    process_tokens(KIND)
    target_zip = ROOT / f'{KIND}_tokens.zip'
    partial_zip = target_zip.with_suffix('.zip.tmp')
    with zipfile.ZipFile(partial_zip, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as z:
        z.write(TOKEN_ROOT / 'manifest.json', 'tokens/manifest.json')
        z.write(TOKEN_ROOT / KIND / 'complete.json', f'tokens/{KIND}/complete.json')
        for start, size in intervals(KIND):
            part = part_path(TOKEN_ROOT / KIND, start)
            z.write(part, f'tokens/{KIND}/{part.name}')
    with zipfile.ZipFile(partial_zip) as z:
        if (bad := z.testzip()) is not None:
            raise IOError(f'Token ZIP integrity error: {bad}')
    partial_zip.replace(target_zip)
    print('Token ZIP:', target_zip)

### Load and validate a compatible token cache

In [ ]:
if MODE == 'build':
    # Read the original token cache, either from a shared ZIP or a local Drive copy.
    # No source CSV or model weights are downloaded here.
    plan = intervals(KIND)
    folder = VECTOR_ROOT/KIND
    folder.mkdir(parents=True, exist_ok=True)
    final_path = VECTOR_ROOT/(KIND+'.npz')
    final_marker = folder/'complete.json'
    archive_path = LOCAL/'digikala_tokens.zip'

    if not LOCAL_TOKEN_ZIP.strip() and not TOKEN_SHARE_LINK.strip():
        generated = ROOT / f'{KIND}_tokens.zip'
        if generated.is_file():
            LOCAL_TOKEN_ZIP = str(generated)

    if LOCAL_TOKEN_ZIP.strip():
        p=Path(LOCAL_TOKEN_ZIP)
        if not p.is_file(): raise FileNotFoundError(p)
        if p.resolve()!=archive_path.resolve():
            if not archive_path.exists() or archive_path.stat().st_size!=p.stat().st_size:
                shutil.copyfile(p,archive_path)
    elif TOKEN_SHARE_LINK.strip() and 'PASTE' not in TOKEN_SHARE_LINK:
        if not archive_path.exists() or not zipfile.is_zipfile(archive_path):
            # gdown 6 accepts the original /file/d/.../view link directly.
            subprocess.run(['gdown','--continue','--retries','3',TOKEN_SHARE_LINK,'-O',str(archive_path)],check=True)
    elif (ROOT/'digikala_tokens.zip').is_file():
        p=ROOT/'digikala_tokens.zip'
        if not archive_path.exists() or archive_path.stat().st_size!=p.stat().st_size:
            shutil.copyfile(p,archive_path)
    else:
        raise ValueError('Paste the public ZIP link in TOKEN_SHARE_LINK, or set LOCAL_TOKEN_ZIP.')

    if not zipfile.is_zipfile(archive_path):
        raise RuntimeError('Invalid ZIP. Check the share permissions / download quota; delete the local broken ZIP and retry.')

    def archive_json(z,name):
        return json.loads(z.read(name).decode('utf-8'))

    def archive_part(z,s,n):
        # zipfile verifies member CRC on read; NPZ loader verifies its own member CRCs.
        with io.BytesIO(z.read(f'tokens/{KIND}/part_{s:09d}.npz')) as b:
            return token_arrays(b,s,n,MAX_TOKENS[KIND])

    with zipfile.ZipFile(archive_path) as z:
        names=z.namelist()
        if len(names)!=len(set(names)): raise ValueError('ZIP has duplicate member names')
        token_manifest=archive_json(z,'tokens/manifest.json')
        token_report=archive_json(z,f'tokens/{KIND}/complete.json')
        TOKEN_SIGNATURE=token_manifest['signature']
        if digest(token_manifest['config'])!=TOKEN_SIGNATURE: raise ValueError('Invalid token manifest signature')
        actual=token_manifest['config']
        expected=TOKEN_CONFIG.copy()
        # Token IDs already exist. Preserve the original tokenizer versions rather than
        # pretending that the current runtime created them. Parity is tested separately.
        expected['tokenizer_versions']=actual['tokenizer_versions']
        if actual!=expected: raise RuntimeError('Wrong token cache settings or dataset/model revision')
        TOKEN_CONFIG=actual
        if (token_report['signature']!=TOKEN_SIGNATURE or token_report['kind']!=KIND or
            not token_report['complete'] or token_report['total_rows']!=SPECS[KIND]['rows']):
            raise RuntimeError('Incomplete/incompatible token cache')
        wanted={f'tokens/{KIND}/part_{s:09d}.npz' for s,n in plan}
        found={name for name in names if name.startswith(f'tokens/{KIND}/part_') and name.endswith('.npz')}
        if wanted!=found: raise RuntimeError('Missing or unexpected token shards')
        token_bytes=sum(z.getinfo(name).file_size for name in wanted)
    print('Selected dataset:', KIND, 'token archive GB:',round(token_bytes/1e9,3))

    # Local archive is always on /content. Only the selected dataset is decoded into RAM.
    # Fall back to local disk reads if the RAM reserve cannot be maintained.
    TOKEN_RAM={}
    if CACHE_TOKENS_IN_RAM and available_ram()>token_bytes+RAM_RESERVE+int(.75*2**30):
        with zipfile.ZipFile(archive_path) as z:
            for s,n in tqdm(plan,desc='Cache selected tokens in RAM',unit='shard'):
                if available_ram()<RAM_RESERVE+int(.75*2**30):
                    print('RAM guard: dropping preload; use local ZIP instead'); TOKEN_RAM.clear(); break
                TOKEN_RAM[s]=archive_part(z,s,n)
    print('RAM cached shards:',len(TOKEN_RAM),'/',len(plan),'available GiB:',round(available_ram()/2**30,2))

### Check configuration compatibility and any completed output

In [ ]:
if MODE == 'build':
    import gc
    import torch
    from sentence_transformers import SentenceTransformer
    from concurrent.futures import ThreadPoolExecutor
    VECTOR_CONFIG=dict(format_version=1,token_signature=TOKEN_SIGNATURE,token_config=TOKEN_CONFIG,
        native_dimension=1024,dimension=DIM,inference_precision='FP16',vector_precision='INT8',
        dimension_reduction='first 768 coordinates then L2 normalize in FP32',
        quantization='per-row symmetric round clip [-127,127], float32 scale',
        empty_text='zero vector, scale=1, valid=False',sentence_transformers=version('sentence-transformers'))
    VECTOR_SIGNATURE=ensure_manifest(VECTOR_ROOT,VECTOR_CONFIG)

    # Existing checkpoints from notebook 02 are accepted only with the exact same vector manifest.
    DONE=False
    if final_marker.exists() and final_path.is_file():
        r=read_json(final_marker)
        if (r.get('complete') and r.get('signature')==VECTOR_SIGNATURE and
            r.get('total_rows')==SPECS[KIND]['rows'] and r.get('final_bytes')==final_path.stat().st_size and
            r.get('final_sha256')):
            DONE=sha256_file(final_path)==r['final_sha256']
            if DONE: print('Verified completed final file:',final_path)

### Load Yalda on the GPU

In [ ]:
if MODE == 'build':
    if not DONE:
        assert torch.cuda.is_available(), 'Switch Colab runtime to GPU (T4).'
        model=SentenceTransformer(MODEL_ID,revision=MODEL_REVISION,device='cuda',model_kwargs={'torch_dtype':torch.float16})
        model.eval(); model.tokenizer.padding_side='left'
        assert model.get_sentence_embedding_dimension()==1024
        assert model.prompts['document']==DOCUMENT_PROMPT and model.prompts['query']==QUERY_PROMPT
        assert not getattr(model[0],'do_lower_case',False)
        assert model.tokenizer.truncation_side=='right'
        backbone=model[0].auto_model
        if hasattr(backbone.config,'use_cache'): backbone.config.use_cache=False
        PAD_ID=model.tokenizer.pad_token_id
        assert PAD_ID is not None
        print(torch.cuda.get_device_name(0), 'attention:',getattr(backbone.config,'_attn_implementation',None))
        print('Vector output:',VECTOR_ROOT)

        # The T4 A/B test favored 128 rows. Benchmark token-budget schedules instead of maximizing VRAM.
        TUNE_ROWS=1536
        CANDIDATES=[(64,4096),(128,8192),(128,16384),(256,8192),(256,16384),(512,16384),(512,32768),(1024,32768),(1024,65536),(2048,65536)]
        DEFAULT=(128,8192)
        AUTO_TUNE=True
        PREFETCH=True
        VRAM_HEADROOM=.90

### PyTorch inference and batch tuning

In [ ]:
if MODE == 'build':
    if not DONE:
        def unit_rows(x):
            x=np.asarray(x,dtype=np.float32)
            norms=np.linalg.norm(x,axis=1,keepdims=True)
            if not np.isfinite(x).all() or np.any(norms<=1e-12): raise ValueError('Invalid embedding')
            return x/norms

        def quantize(x):
            scales=np.maximum(np.max(np.abs(x),axis=1)/127,1e-12).astype(np.float32)
            codes=np.clip(np.rint(x/scales[:,None]),-127,127).astype(np.int8)
            return codes,scales

        def reconstruct(codes,scales):
            return unit_rows(codes.astype(np.float32)*scales[:,None])

        def batches(seqs,max_rows,budget):
            # Sort by actual token length and cap padded token count per batch.
            order=sorted(range(len(seqs)),key=lambda i:len(seqs[i]))
            begin=0
            while begin<len(order):
                end=begin+1
                while end<len(order) and end-begin<max_rows and (end-begin+1)*len(seqs[order[end]])<=budget:
                    end+=1
                yield order[begin:end]
                begin=end

        @torch.inference_mode()
        def encode_ids(seqs,settings):
            if not seqs: return np.empty((0,DIM),dtype=np.float32)
            result=torch.empty((len(seqs),DIM),dtype=torch.float32,device='cuda')
            for inds in batches(seqs,*settings):
                length=max(len(seqs[i]) for i in inds)
                ids=np.full((len(inds),length),PAD_ID,dtype=np.int64)
                mask=np.zeros_like(ids)
                for j,i in enumerate(inds):
                    s=seqs[i]; ids[j,-len(s):]=s; mask[j,-len(s):]=1
                features={'input_ids':torch.from_numpy(ids).to('cuda'), 'attention_mask':torch.from_numpy(mask).to('cuda')}
                # Preserve the full SentenceTransformer module chain, including its original pooling.
                out=model(features)['sentence_embedding']
                if out.shape!=(len(inds),1024): raise ValueError(f'Unexpected shape {out.shape}')
                result[torch.as_tensor(inds,device='cuda')]=out[:,:DIM].float()
            return unit_rows(result.cpu().numpy())

        def release():
            gc.collect(); torch.cuda.empty_cache()

        def encode_safe(seqs,settings):
            while True:
                try: return encode_ids(seqs,settings),settings
                except torch.cuda.OutOfMemoryError:
                    if settings[0]==1 and settings[1]<=512: raise
                release(); settings=(max(1,settings[0]//2),max(512,settings[1]//2))
                print('OOM fallback:',settings)

        # Parity check: pretokenized forward must match the usual text-based Yalda encode path.
        # Tests left padding, pooling, truncation and 768-dimensional reduction on real Persian examples.
        probe=['گوشی موبایل سامسونگ | Samsung | گوشی موبایل','کیف مناسب مدرسه','کیفیت کالا خوب نیست. '+'متن آزمایشی طولانی برای بررسی عملکرد مدل. '*150]
        for kind,limit in [(KIND,MAX_TOKENS[KIND])]:
            model.max_seq_length=limit
            ids=model.tokenizer(probe,padding=False,truncation=True,max_length=limit,add_special_tokens=True,
                return_attention_mask=False,return_token_type_ids=False)['input_ids']
            seqs=[np.asarray(x,dtype=np.uint32) for x in ids]
            actual=encode_ids(seqs,(16,8192))
            ref=model.encode(probe,prompt_name='document',batch_size=16,convert_to_numpy=True,normalize_embeddings=False,show_progress_bar=False)
            ref=unit_rows(ref[:,:DIM])
            cosine=np.sum(actual*ref,axis=1)
            print('Parity',kind,cosine)
            if np.min(cosine)<.999: raise RuntimeError('Pretokenized forward differs from reference encode; stop before production.')
        release()

        def tune(seqs):
            if not AUTO_TUNE or not seqs:return DEFAULT,[]
            rng=np.random.default_rng(42)
            subset=[seqs[i] for i in rng.choice(len(seqs),min(TUNE_ROWS,len(seqs)),replace=False)]
            _,_=encode_safe(subset[:min(64,len(subset))],DEFAULT)
            reports=[]; capacity=torch.cuda.get_device_properties(0).total_memory
            for setting in CANDIDATES:
                release(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t=time.perf_counter()
                try:
                    _=encode_ids(subset,setting); torch.cuda.synchronize()
                    sec=time.perf_counter()-t; peak=torch.cuda.max_memory_reserved()
                    reports.append(dict(rows=setting[0],budget=setting[1],rows_s=len(subset)/sec,peak_gib=peak/2**30,safe=peak<=capacity*VRAM_HEADROOM))
                except torch.cuda.OutOfMemoryError:
                    reports.append(dict(rows=setting[0],budget=setting[1],rows_s=0,peak_gib=None,safe=False))
            release(); safe=[r for r in reports if r['safe']]
            best=max(safe,key=lambda r:r['rows_s']) if safe else None
            setting=(best['rows'],best['budget']) if best else (32,4096)
            print('Selected',setting, 'benchmark',reports)
            return setting,reports

        def unique_sequences(ids,offsets,valid):
            # Identical truncated token sequences give the same input; keep every original index.
            unique=[]; mapping={}; inverse=[]
            for i in np.flatnonzero(valid):
                seq=ids[offsets[i]:offsets[i+1]]; key=seq.tobytes()
                j=mapping.get(key)
                if j is None:
                    j=len(unique); mapping[key]=j; unique.append(seq)
                inverse.append(j)
            return unique,np.asarray(inverse,dtype=np.int32)

### Write the checkpoints and final NPZ

In [ ]:
if MODE == 'build':
    def vector_arrays(path,start,size,return_arrays=False):
        with np.load(path,allow_pickle=False) as d:
            if set(d.files)!={'source_row','vectors','scales','valid'}: raise ValueError('Invalid vector schema')
            rows,codes,scales,valid=(d[k] for k in ('source_row','vectors','scales','valid'))
        if rows.dtype!=np.int64 or not np.array_equal(rows,np.arange(start,start+size,dtype=np.int64)): raise ValueError('Invalid row index')
        if codes.dtype!=np.int8 or codes.shape!=(size,DIM) or np.any(codes==-128): raise ValueError('Invalid codes')
        if scales.dtype!=np.float32 or scales.shape!=(size,) or not np.isfinite(scales).all() or np.any(scales<=0): raise ValueError('Invalid scales')
        if valid.dtype!=np.bool_ or valid.shape!=(size,): raise ValueError('Invalid valid flags')
        if np.any(codes[~valid]!=0) or np.any(scales[~valid]!=1) or np.any(~np.any(codes[valid]!=0,axis=1)): raise ValueError('Invalid placeholders')
        return (codes,scales,valid) if return_arrays else int(valid.sum())



    def make_local_arrays(total):
        # Local disk-backed vectors avoid a 4.8 GB permanent allocation; Linux can
        # cache hot pages in spare RAM. This is not persistent Drive checkpointing.
        return (np.lib.format.open_memmap(LOCAL/'vectors_work.npy',mode='w+',dtype=np.int8,shape=(total,DIM)),
                np.lib.format.open_memmap(LOCAL/'scales_work.npy',mode='w+',dtype=np.float32,shape=(total,)),
                np.lib.format.open_memmap(LOCAL/'valid_work.npy',mode='w+',dtype=np.bool_,shape=(total,)))

    def read_existing(path,s,n,valid):
        codes,scales,stored_valid=vector_arrays(path,s,n,return_arrays=True)
        if not np.array_equal(stored_valid,valid): raise ValueError('Checkpoint validity mismatch')
        return codes,scales

    def canonicalize(ids,offsets,valid,start,global_map):
        # Python bytes dict compares full keys, not only hashes. Distinct sequences
        # never share a vector due to a hash collision. Local dedup stays enabled.
        canonical=np.empty(int(valid.sum()),dtype=np.int64)
        local_map={}; reps=[]; seqs=[]
        for j,i in enumerate(np.flatnonzero(valid)):
            seq=ids[offsets[i]:offsets[i+1]]; key=seq.tobytes()
            rep=global_map.get(key)
            if rep is None:
                rep=local_map.get(key)
                if rep is None:
                    rep=start+int(i); local_map[key]=rep; reps.append(rep); seqs.append(seq)
            canonical[j]=rep
        return canonical,local_map,np.asarray(reps,dtype=np.int64),seqs

    def add_global(global_map,local_map,enabled):
        if not enabled: return False
        if available_ram()<RAM_RESERVE:
            print('RAM guard reached: stop adding global keys; keep existing global hits and local dedup.')
            return False
        global_map.update(local_map)
        return True

    def finalize(codes,scales,valid,report):
        # A single conventional NPZ per dataset, in original row order.
        # Checkpoints stay on Drive until the final copy is verified.
        temp=LOCAL/(KIND+'_final.npz')
        rows=np.arange(len(valid),dtype=np.int64)
        np.savez(temp,source_row=rows,vectors=codes,scales=scales,valid=valid)
        del rows
        with zipfile.ZipFile(temp) as z:
            assert set(z.namelist())=={'source_row.npy','vectors.npy','scales.npy','valid.npy'}
            bad=z.testzip()
            if bad is not None: raise IOError('Local NPZ CRC error: '+bad)
        h=sha256_file(temp)
        tmp=final_path.with_suffix('.npz.tmp')
        shutil.copyfile(temp,tmp)
        if tmp.stat().st_size!=temp.stat().st_size or sha256_file(tmp)!=h:
            raise IOError('Final Drive copy failed verification; checkpoint shards retained')
        tmp.replace(final_path)
        report.update(final_bytes=final_path.stat().st_size,final_sha256=h,final_file=final_path.name,complete=True)
        write_json(final_marker,report)
        if not KEEP_CHECKPOINTS:
            for s,n in plan: part_path(folder,s).unlink(missing_ok=True)
        temp.unlink(missing_ok=True)
        print('Final verified NPZ:',final_path)


    def process():
        total=SPECS[KIND]['rows']; codes,scales,valid_store=make_local_arrays(total)
        # A map of full token sequence -> canonical original row (vector cache is local).
        global_map={}; accept_new=GLOBAL_DEDUP
        settings=DEFAULT; tuned=False; reports=[]
        n_new=n_resumed=n_encoded=n_valid=0; n_global_hits=n_local_hits=0
        encode_sec=save_sec=0.; t0=time.perf_counter()
        final_marker.unlink(missing_ok=True)
        def get_part(z,s,n):
            return TOKEN_RAM[s] if s in TOKEN_RAM else archive_part(z,s,n)
        with zipfile.ZipFile(archive_path) as z, tqdm(total=total,desc='Embed '+KIND,unit='rows') as bar:
            for start,size in plan:
                end=start+size; ids,offsets,valid=get_part(z,start,size); path=part_path(folder,start)
                valid_store[start:end]=valid; n_valid+=int(valid.sum())
                old=None
                if path.exists():
                    try: old=read_existing(path,start,size,valid)
                    except Exception as e: print('Rebuilding invalid checkpoint',path.name,str(e))
                if old is not None:
                    # Reconstruct global cache across restarts from completed shards.
                    codes[start:end],scales[start:end]=old
                    if accept_new:
                        _,local_map,_,_=canonicalize(ids,offsets,valid,start,global_map)
                        accept_new=add_global(global_map,local_map,accept_new)
                    n_resumed+=size
                else:
                    canonical,local_map,reps,seqs=canonicalize(ids,offsets,valid,start,global_map)
                    n_global_hits+=int(np.count_nonzero(canonical<start))
                    n_local_hits+=int(len(canonical)-np.count_nonzero(canonical<start)-len(seqs))
                    if seqs and not tuned:
                        # Benchmark first unprocessed unique shard; sorted length-aware batches.
                        settings,reports=tune(seqs); tuned=True
                        write_json(folder/'batch_benchmark_latest.json',dict(gpu=torch.cuda.get_device_name(0),selected=settings,results=reports))
                    if seqs:
                        t=time.perf_counter(); x,settings=encode_safe(seqs,settings); c,s=quantize(x)
                        codes[reps]=c; scales[reps]=s; n_encoded+=len(seqs)
                        encode_sec+=time.perf_counter()-t
                    # Gather first, then write; some representatives are in this same shard.
                    block=np.zeros((size,DIM),np.int8); scale_block=np.ones(size,np.float32)
                    if len(canonical): block[valid]=codes[canonical]; scale_block[valid]=scales[canonical]
                    codes[start:end]=block; scales[start:end]=scale_block
                    t=time.perf_counter()
                    publish(path,dict(source_row=np.arange(start,end,dtype=np.int64),vectors=block,scales=scale_block,valid=valid),vector_arrays,start,size)
                    save_sec+=time.perf_counter()-t
                    accept_new=add_global(global_map,local_map,accept_new)
                    n_new+=size
                bar.update(size)
                sec=time.perf_counter()-t0
                bar.set_postfix(new=n_new,resumed=n_resumed,encoded=n_encoded,global_hits=n_global_hits,
                                batch=settings[0],budget=settings[1],wall_rows_s=round(n_new/max(sec,1e-9),1),
                                unique_gpu_s=round(n_encoded/max(encode_sec,1e-9),1),ram_free_GiB=round(available_ram()/2**30,1))
        check_parts(folder,KIND)
        if n_valid!=token_report['valid_rows'] or n_new+n_resumed!=total:
            raise RuntimeError('Row/validity count mismatch')
        report=dict(signature=VECTOR_SIGNATURE,token_signature=TOKEN_SIGNATURE,kind=KIND,total_rows=total,
                    valid_vectors=n_valid,new_rows=n_new,resumed_rows=n_resumed,unique_sequences_encoded=n_encoded,
                    global_hits=n_global_hits,local_hits=n_local_hits,global_cache_keys=len(global_map),
                    global_cache_enabled=GLOBAL_DEDUP,global_cache_fully_enabled=bool(accept_new),
                    encoding_seconds=encode_sec,storage_seconds=save_sec,wall_seconds=time.perf_counter()-t0,
                    last_schedule=settings,complete=False)
        # Release cache before making final archive.
        global_map.clear(); TOKEN_RAM.clear(); gc.collect()
        codes.flush(); scales.flush(); valid_store.flush()
        finalize(codes,scales,valid_store,report)
        print(json.dumps(report,indent=2))
        del codes,scales,valid_store
        for name in ('vectors_work.npy','scales_work.npy','valid_work.npy'):
            (LOCAL/name).unlink(missing_ok=True)

### Build or resume

In [ ]:
if MODE == 'build':
    if not DONE:
        process()
    else:
        if not KEEP_CHECKPOINTS:
            for s,n in plan: part_path(folder,s).unlink(missing_ok=True)
        print("Already complete; no embedding required.")

### Output files

The finished `vectors/products.npz` is ready for use in section 2. Keep `vectors/manifest.json` and `vectors/products/complete.json` for reproducibility and resuming. With `KEEP_CHECKPOINTS = False`, the notebook removes intermediate vector shards after the final file is saved and verified.